In [ ]:
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import os, sys
from matplotlib.gridspec import GridSpec
from matplotlib.animation import FuncAnimation
from matplotlib.ticker import AutoMinorLocator
from IPython.display import HTML

sys.path.insert(0, os.path.abspath('../code'))
from axion_solver import Params
from greens_solver import StepParams, E_tilde

plt.style.use('mine.mplstyle')
plt.rcParams["axes.axisbelow"] = False
plt.rcParams.update({
    "text.usetex": True,
    "font.family": "serif",
    "text.latex.preamble": r"\usepackage{amssymb}",
    "animation.html": "html5",
    "animation.embed_limit": 256,
})

p = Params()
sp = StepParams()
L = p.L_region
Lb = p.L_box
ma = p.m_a
eps = 0.015
edges = [0.0, L, 2*L, 3*L, 4*L, 5*L]

In [ ]:
# Figure 1: the time-domain (ODE) solver against the frequency-domain Green's
# function, same piecewise-constant profile, with their difference underneath.
gr = np.load('../data/greens_run.npz')
x_gr = gr['x']
t_gr = gr['t']
E_gr = gr['E'] / p.gB0a0
E_td = gr['E_td'] / p.gB0a0
dE_gr = E_gr - E_td

step_gr = 2                      # every 2nd snapshot -> 301 frames, 12 s at 25 fps
frames_gr = np.arange(0, len(t_gr), step_gr)

xf_gr = np.linspace(x_gr[0], x_gr[-1], 4000)
wp_gr = sp.omega_p(xf_gr) / ma
ymax_gr = 1.08 * max(np.abs(E_gr).max(), np.abs(E_td).max())
dmax_gr = 1.15 * np.abs(dE_gr).max()
cs_solvers = ['tab:blue', 'tab:red']

fig1 = plt.figure(figsize=(7.9, 7.9 / 1.61803398875 * 1.15), dpi=110)
gs1 = GridSpec(26, 40, figure=fig1)
ax_wp = fig1.add_subplot(gs1[0:5, 1:-1])
ax_Ey = fig1.add_subplot(gs1[6:19, 1:-1])
ax_dE = fig1.add_subplot(gs1[20:26, 1:-1])

ax_wp.set_xlim(x_gr[0], x_gr[-1])
ax_wp.set_ylim(0.70, 1.34)
ax_wp.set_ylabel(r'$\omega_p/m_a$', labelpad=2)
ax_wp.set_xticklabels([])
ax_wp.axvspan(edges[2], edges[3], color='0.88')
ax_wp.axhline(1.0, lw=0.5, ls='--', c='0.45')
ax_wp.plot(xf_gr, wp_gr, lw=0.8, c='k')

ax_Ey.set_xlim(x_gr[0], x_gr[-1])
ax_Ey.set_ylim(-ymax_gr, ymax_gr)
ax_Ey.set_ylabel(r'$E_y\,/\,(g_{a\gamma\gamma}B_0a_0)$', labelpad=2)
ax_Ey.set_xticklabels([])
ax_Ey.axvspan(edges[2], edges[3], color='0.88')
ax_Ey.axhline(0.0, lw=0.4, c='0.8')

ax_dE.set_xlim(x_gr[0], x_gr[-1])
ax_dE.set_ylim(-dmax_gr, dmax_gr)
ax_dE.set_xlabel(r'$x$')
ax_dE.set_ylabel(r'$\Delta E_y$', labelpad=2)
ax_dE.axvspan(edges[2], edges[3], color='0.88')
ax_dE.axhline(0.0, lw=0.4, c='0.8')

for xe in edges:
    ax_wp.axvline(xe, lw=0.4, ls=':', c='0.6')
    ax_Ey.axvline(xe, lw=0.4, ls=':', c='0.6')
    ax_dE.axvline(xe, lw=0.4, ls=':', c='0.6')

l_td, = ax_Ey.plot([], [], lw=0.9, c=cs_solvers[0], label=r'time-domain solver')
l_gr, = ax_Ey.plot([], [], lw=0.9, ls='--', c=cs_solvers[1], label=r"Green's function")
l_dE, = ax_dE.plot([], [], lw=0.6, c='0.35')
txt_gr = ax_Ey.text(0.985, 0.92, '', transform=ax_Ey.transAxes, ha='right',
                    va='top', fontsize=8, usetex=False)
ax_Ey.legend(loc='upper left')
ax_wp.set_title(r'axion wavepacket through five plasma slabs: two independent solutions',
                fontsize=8)

def update_gr(j):
    l_td.set_data(x_gr, E_td[j])
    l_gr.set_data(x_gr, E_gr[j])
    l_dE.set_data(x_gr, dE_gr[j])
    txt_gr.set_text('t = %6.1f' % t_gr[j])
    return l_td, l_gr, l_dE, txt_gr

ani1 = FuncAnimation(fig1, update_gr, frames=frames_gr, interval=40, blit=False)
plt.close(fig1)
HTML(ani1.to_html5_video())

In [ ]:
# Figure 2: five single-frequency modes, Re[E~ e^{-i phi}], animated against the
# mode's own phase so all five rows stay locked together. One shared absolute
# y-scale across the rows, so the panel heights compare the actual response.
oms_modes = np.array([0.85, 0.95, 1.00, 1.05, 1.15])
cs_modes = ['tab:cyan', 'tab:blue', 'k', 'tab:red', 'tab:orange']
x_modes = np.linspace(-3.0, 55.0, 1200)
Et_modes = E_tilde(x_modes, oms_modes + 1j*eps, p) / p.gB0a0
ylim_modes = 1.06 * np.abs(Et_modes).max()

xf_modes = np.linspace(x_modes[0], x_modes[-1], 4000)
wp_modes = sp.omega_p(xf_modes) / ma
phases = np.linspace(0.0, 4*np.pi, 180, endpoint=False)

fig2 = plt.figure(figsize=(7.9, 7.9 / 1.61803398875 * 2.05), dpi=110)
gs2 = GridSpec(31, 40, figure=fig2)
ax2_wp = fig2.add_subplot(gs2[0:5, 1:-1])
ax2_a = fig2.add_subplot(gs2[6:10, 1:-1])
ax2_b = fig2.add_subplot(gs2[11:15, 1:-1])
ax2_c = fig2.add_subplot(gs2[16:20, 1:-1])
ax2_d = fig2.add_subplot(gs2[21:25, 1:-1])
ax2_e = fig2.add_subplot(gs2[26:30, 1:-1])

ax2_wp.set_xlim(x_modes[0], x_modes[-1])
ax2_wp.set_ylim(0.74, 1.30)
ax2_wp.set_ylabel(r'$\omega_p/m_a$', labelpad=2)
ax2_wp.set_xticklabels([])
ax2_wp.axvspan(edges[2], edges[3], color='0.88')
ax2_wp.plot(xf_modes, wp_modes, lw=0.9, c='k')
ax2_wp.axhline(oms_modes[0], lw=0.6, c=cs_modes[0])
ax2_wp.axhline(oms_modes[1], lw=0.6, c=cs_modes[1])
ax2_wp.axhline(oms_modes[2], lw=0.6, c=cs_modes[2])
ax2_wp.axhline(oms_modes[3], lw=0.6, c=cs_modes[3])
ax2_wp.axhline(oms_modes[4], lw=0.6, c=cs_modes[4])
ax2_wp.set_title(r'a mode propagates only where its line sits above the staircase', fontsize=8)

ax2_a.set_xlim(x_modes[0], x_modes[-1])
ax2_a.set_ylim(-ylim_modes, ylim_modes)
ax2_a.set_ylabel(r'$\omega=0.85$', labelpad=2, fontsize=8)
ax2_a.set_xticklabels([])
ax2_a.axvspan(edges[2], edges[3], color='0.88')
ax2_a.fill_between(x_modes, -np.abs(Et_modes[0]), np.abs(Et_modes[0]), color=cs_modes[0], alpha=0.15, lw=0)
l2_a, = ax2_a.plot([], [], lw=1.0, c=cs_modes[0])

ax2_b.set_xlim(x_modes[0], x_modes[-1])
ax2_b.set_ylim(-ylim_modes, ylim_modes)
ax2_b.set_ylabel(r'$\omega=0.95$', labelpad=2, fontsize=8)
ax2_b.set_xticklabels([])
ax2_b.axvspan(edges[2], edges[3], color='0.88')
ax2_b.fill_between(x_modes, -np.abs(Et_modes[1]), np.abs(Et_modes[1]), color=cs_modes[1], alpha=0.15, lw=0)
l2_b, = ax2_b.plot([], [], lw=1.0, c=cs_modes[1])

ax2_c.set_xlim(x_modes[0], x_modes[-1])
ax2_c.set_ylim(-ylim_modes, ylim_modes)
ax2_c.set_ylabel(r'$\omega=m_a$', labelpad=2, fontsize=8)
ax2_c.set_xticklabels([])
ax2_c.axvspan(edges[2], edges[3], color='0.88')
ax2_c.fill_between(x_modes, -np.abs(Et_modes[2]), np.abs(Et_modes[2]), color=cs_modes[2], alpha=0.15, lw=0)
l2_c, = ax2_c.plot([], [], lw=1.0, c=cs_modes[2])

ax2_d.set_xlim(x_modes[0], x_modes[-1])
ax2_d.set_ylim(-ylim_modes, ylim_modes)
ax2_d.set_ylabel(r'$\omega=1.05$', labelpad=2, fontsize=8)
ax2_d.set_xticklabels([])
ax2_d.axvspan(edges[2], edges[3], color='0.88')
ax2_d.fill_between(x_modes, -np.abs(Et_modes[3]), np.abs(Et_modes[3]), color=cs_modes[3], alpha=0.15, lw=0)
l2_d, = ax2_d.plot([], [], lw=1.0, c=cs_modes[3])

ax2_e.set_xlim(x_modes[0], x_modes[-1])
ax2_e.set_ylim(-ylim_modes, ylim_modes)
ax2_e.set_ylabel(r'$\omega=1.15$', labelpad=2, fontsize=8)
ax2_e.set_xlabel(r'$x$')
ax2_e.axvspan(edges[2], edges[3], color='0.88')
ax2_e.fill_between(x_modes, -np.abs(Et_modes[4]), np.abs(Et_modes[4]), color=cs_modes[4], alpha=0.15, lw=0)
l2_e, = ax2_e.plot([], [], lw=1.0, c=cs_modes[4])

fig2.text(0.045, 0.42, r'$\mathrm{Re}\,[\,\tilde{E}_y(x,\omega)\,e^{-i\phi}\,]\,/\,(g_{a\gamma\gamma}B_0a_0)$',
          rotation=90, va='center', ha='center')
txt_ph = ax2_wp.text(0.985, 0.88, '', transform=ax2_wp.transAxes, ha='right',
                     va='top', fontsize=8, usetex=False)

def update_phase(k):
    l2_a.set_data(x_modes, (Et_modes[0]*np.exp(-1j*phases[k])).real)
    l2_b.set_data(x_modes, (Et_modes[1]*np.exp(-1j*phases[k])).real)
    l2_c.set_data(x_modes, (Et_modes[2]*np.exp(-1j*phases[k])).real)
    l2_d.set_data(x_modes, (Et_modes[3]*np.exp(-1j*phases[k])).real)
    l2_e.set_data(x_modes, (Et_modes[4]*np.exp(-1j*phases[k])).real)
    txt_ph.set_text('phase = %4.2f' % (phases[k] % (2*np.pi)))
    return l2_a, l2_b, l2_c, l2_d, l2_e, txt_ph

ani2 = FuncAnimation(fig2, update_phase, frames=len(phases), interval=42, blit=False)
plt.close(fig2)
HTML(ani2.to_html5_video())

In [ ]:
# Figure 3: the same five modes against real time. phi = omega t, so each row
# now runs at its own rate: they start aligned at t = 0 and drift apart, which
# is the 35% spread in omega made visible. Reuses Et_modes from the cell above.
times = np.linspace(0.0, 30.0, 200, endpoint=False)

fig3 = plt.figure(figsize=(7.9, 7.9 / 1.61803398875 * 2.05), dpi=110)
gs3 = GridSpec(31, 40, figure=fig3)
ax3_wp = fig3.add_subplot(gs3[0:5, 1:-1])
ax3_a = fig3.add_subplot(gs3[6:10, 1:-1])
ax3_b = fig3.add_subplot(gs3[11:15, 1:-1])
ax3_c = fig3.add_subplot(gs3[16:20, 1:-1])
ax3_d = fig3.add_subplot(gs3[21:25, 1:-1])
ax3_e = fig3.add_subplot(gs3[26:30, 1:-1])

ax3_wp.set_xlim(x_modes[0], x_modes[-1])
ax3_wp.set_ylim(0.74, 1.30)
ax3_wp.set_ylabel(r'$\omega_p/m_a$', labelpad=2)
ax3_wp.set_xticklabels([])
ax3_wp.axvspan(edges[2], edges[3], color='0.88')
ax3_wp.plot(xf_modes, wp_modes, lw=0.9, c='k')
ax3_wp.axhline(oms_modes[0], lw=0.6, c=cs_modes[0])
ax3_wp.axhline(oms_modes[1], lw=0.6, c=cs_modes[1])
ax3_wp.axhline(oms_modes[2], lw=0.6, c=cs_modes[2])
ax3_wp.axhline(oms_modes[3], lw=0.6, c=cs_modes[3])
ax3_wp.axhline(oms_modes[4], lw=0.6, c=cs_modes[4])
ax3_wp.set_title(r'the same five modes in real time: each row runs at its own $\omega$', fontsize=8)

ax3_a.set_xlim(x_modes[0], x_modes[-1])
ax3_a.set_ylim(-ylim_modes, ylim_modes)
ax3_a.set_ylabel(r'$\omega=0.85$', labelpad=2, fontsize=8)
ax3_a.set_xticklabels([])
ax3_a.axvspan(edges[2], edges[3], color='0.88')
ax3_a.fill_between(x_modes, -np.abs(Et_modes[0]), np.abs(Et_modes[0]), color=cs_modes[0], alpha=0.15, lw=0)
l3_a, = ax3_a.plot([], [], lw=1.0, c=cs_modes[0])

ax3_b.set_xlim(x_modes[0], x_modes[-1])
ax3_b.set_ylim(-ylim_modes, ylim_modes)
ax3_b.set_ylabel(r'$\omega=0.95$', labelpad=2, fontsize=8)
ax3_b.set_xticklabels([])
ax3_b.axvspan(edges[2], edges[3], color='0.88')
ax3_b.fill_between(x_modes, -np.abs(Et_modes[1]), np.abs(Et_modes[1]), color=cs_modes[1], alpha=0.15, lw=0)
l3_b, = ax3_b.plot([], [], lw=1.0, c=cs_modes[1])

ax3_c.set_xlim(x_modes[0], x_modes[-1])
ax3_c.set_ylim(-ylim_modes, ylim_modes)
ax3_c.set_ylabel(r'$\omega=m_a$', labelpad=2, fontsize=8)
ax3_c.set_xticklabels([])
ax3_c.axvspan(edges[2], edges[3], color='0.88')
ax3_c.fill_between(x_modes, -np.abs(Et_modes[2]), np.abs(Et_modes[2]), color=cs_modes[2], alpha=0.15, lw=0)
l3_c, = ax3_c.plot([], [], lw=1.0, c=cs_modes[2])

ax3_d.set_xlim(x_modes[0], x_modes[-1])
ax3_d.set_ylim(-ylim_modes, ylim_modes)
ax3_d.set_ylabel(r'$\omega=1.05$', labelpad=2, fontsize=8)
ax3_d.set_xticklabels([])
ax3_d.axvspan(edges[2], edges[3], color='0.88')
ax3_d.fill_between(x_modes, -np.abs(Et_modes[3]), np.abs(Et_modes[3]), color=cs_modes[3], alpha=0.15, lw=0)
l3_d, = ax3_d.plot([], [], lw=1.0, c=cs_modes[3])

ax3_e.set_xlim(x_modes[0], x_modes[-1])
ax3_e.set_ylim(-ylim_modes, ylim_modes)
ax3_e.set_ylabel(r'$\omega=1.15$', labelpad=2, fontsize=8)
ax3_e.set_xlabel(r'$x$')
ax3_e.axvspan(edges[2], edges[3], color='0.88')
ax3_e.fill_between(x_modes, -np.abs(Et_modes[4]), np.abs(Et_modes[4]), color=cs_modes[4], alpha=0.15, lw=0)
l3_e, = ax3_e.plot([], [], lw=1.0, c=cs_modes[4])

fig3.text(0.045, 0.42, r'$\mathrm{Re}\,[\,\tilde{E}_y(x,\omega)\,e^{-i\omega t}\,]\,/\,(g_{a\gamma\gamma}B_0a_0)$',
          rotation=90, va='center', ha='center')
txt_tm = ax3_wp.text(0.985, 0.88, '', transform=ax3_wp.transAxes, ha='right',
                     va='top', fontsize=8, usetex=False)

def update_time(k):
    l3_a.set_data(x_modes, (Et_modes[0]*np.exp(-1j*oms_modes[0]*times[k])).real)
    l3_b.set_data(x_modes, (Et_modes[1]*np.exp(-1j*oms_modes[1]*times[k])).real)
    l3_c.set_data(x_modes, (Et_modes[2]*np.exp(-1j*oms_modes[2]*times[k])).real)
    l3_d.set_data(x_modes, (Et_modes[3]*np.exp(-1j*oms_modes[3]*times[k])).real)
    l3_e.set_data(x_modes, (Et_modes[4]*np.exp(-1j*oms_modes[4]*times[k])).real)
    txt_tm.set_text('t = %5.2f' % times[k])
    return l3_a, l3_b, l3_c, l3_d, l3_e, txt_tm

ani3 = FuncAnimation(fig3, update_time, frames=len(times), interval=42, blit=False)
plt.close(fig3)
HTML(ani3.to_html5_video())